In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import kete
from astropy.io import ascii

# example astronomy is generated for (617) Patroclus via JPL Horizons
t=ascii.read('example_astrometry.csv')

observations = []
for i in range(len(t)):
    observer = kete.spice.mpc_code_to_ecliptic(str(t['Obs'][i]), t['JD'][i])
    observations.append(kete.orbit_fitting.Observation.optical(observer = observer, ra=t['RA'][i], dec=t['Dec'][i], sigma_ra = 0.1, sigma_dec = 0.1))

# perform initial orbit determination    
candidates = kete.orbit_fitting.initial_orbit_determination(observations)
# refine on the best candidate orbit
fit = kete.orbit_fitting.fit_orbit(candidates[0][1], observations)

# estimate uncertainties
sample_states, _ = fit.uncertain_state.sample(1000)

a = []
e = []
i = []
node = []
arg = []
tperi = []
for s in sample_states:
    elem = kete.vector.CometElements.from_state(s)
    a.append(elem.semi_major)
    e.append(elem.eccentricity)
    i.append(elem.inclination)
    node.append(elem.lon_of_ascending)
    arg.append(elem.peri_arg)
    tperi.append(elem.peri_time.jd)

a_stat = np.quantile(a, [0.16, 0.5, 0.84])
e_stat = np.quantile(e, [0.16, 0.5, 0.84])
i_stat = np.quantile(i, [0.16, 0.5, 0.84])
node_stat = np.quantile(node, [0.16, 0.5, 0.84])
arg_stat = np.quantile(arg, [0.16, 0.5, 0.84])
tperi_stat = np.quantile(tperi, [0.16, 0.5, 0.84])


# print the result
print('a = %.6f (-%.6f / +%.6f) AU'%(a_stat[1], a_stat[1]-a_stat[0], a_stat[2]-a_stat[1]))
print('e = %.6f (-%.6f / +%.6f)'%(e_stat[1], e_stat[1]-e_stat[0], e_stat[2]-e_stat[1]))
print('i = %.6f (-%.6f / +%.6f) deg'%(i_stat[1], i_stat[1]-i_stat[0], i_stat[2]-i_stat[1]))
print('Longitude of ascending node = %.6f (-%.6f / +%.6f) deg'%(node_stat[1], node_stat[1]-node_stat[0], node_stat[2]-node_stat[1]))
print('Argument of perihelion = %.6f (-%.6f / +%.6f) deg'%(arg_stat[1], arg_stat[1]-arg_stat[0], arg_stat[2]-arg_stat[1]))
print('Time of perihelion = %s (-%.6f / +%.6f) days'%(kete.Time(tperi_stat[1]).iso, tperi_stat[1]-tperi_stat[0], tperi_stat[2]-tperi_stat[1]))


